# Argentina vs England Pre-Match xG Prediction

This notebook trains a local custom xG model from raw StatsBomb shot events, then compares the Argentina-England prediction against StatsBomb's provided `shot.statsbomb_xg`.

The notebook uses local JSON only. It expands the senior men's proxy set to include Copa America for Argentina and Euro for England, while keeping World Cup data for both where available.

In [ ]:

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

ROOT = Path.cwd()
DATA_DIR = ROOT / "archive" / "data"
OUTPUT_DIR = ROOT / "outputs" / "argentina_england_xg_prediction"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEAM_A = "Argentina"
TEAM_B = "England"
MATCH_LABEL = "Argentina vs England"
MATCH_DATE = "2026-07-15"
RECENT_MATCH_COUNT = 10
SHRINKAGE_MATCHES = 8
MAX_GOALS = 10
POST_MATCH_FEEDBACK = "User-provided note: the previous Spain-France prediction was directionally right, with Spain beating France 2-0."
REINFORCED_FEATURE_HINTS = ["angle_to_goal", "freeze_min_opponent_distance", "freeze_opponents_goal_side", "freeze_goalkeeper_distance", "x", "body_part", "open_goal", "play_pattern"]

SELECTED_COMPETITIONS = {
    ("FIFA World Cup", "2018"),
    ("FIFA World Cup", "2022"),
    ("UEFA Euro", "2020"),
    ("UEFA Euro", "2024"),
    ("Copa America", "2021"),
    ("Copa America", "2024"),
}


def get_nested(obj, path, default=np.nan):
    cur = obj
    for part in path:
        if isinstance(cur, dict) and part in cur:
            cur = cur[part]
        else:
            return default
    return cur


def shot_angle(x, y):
    left_post = np.array([120.0, 36.0])
    right_post = np.array([120.0, 44.0])
    point = np.array([float(x), float(y)])
    a = left_post - point
    b = right_post - point
    denom = (np.linalg.norm(a) * np.linalg.norm(b)) or 1.0
    cosang = np.clip(np.dot(a, b) / denom, -1.0, 1.0)
    return float(np.arccos(cosang))


def freeze_features(event):
    frame = get_nested(event, ["shot", "freeze_frame"], default=[])
    if not isinstance(frame, list):
        frame = []
    sx, sy = event.get("location", [np.nan, np.nan])[:2]
    teammates = 0
    opponents = 0
    opponents_goal_side = 0
    min_opp_dist = np.nan
    gk_dist = np.nan
    for item in frame:
        loc = item.get("location") or [np.nan, np.nan]
        if len(loc) < 2 or pd.isna(loc[0]) or pd.isna(loc[1]):
            continue
        dist = float(math.hypot(float(loc[0]) - float(sx), float(loc[1]) - float(sy)))
        if item.get("teammate"):
            teammates += 1
        else:
            opponents += 1
            if float(loc[0]) > float(sx):
                opponents_goal_side += 1
            min_opp_dist = dist if pd.isna(min_opp_dist) else min(min_opp_dist, dist)
            pos_name = get_nested(item, ["position", "name"], default="")
            if str(pos_name).lower() == "goalkeeper":
                gk_dist = dist if pd.isna(gk_dist) else min(gk_dist, dist)
    return teammates, opponents, opponents_goal_side, min_opp_dist, gk_dist


def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def poisson_pmf(lam, max_goals=MAX_GOALS):
    vals = np.array([math.exp(-lam) * lam ** k / math.factorial(k) for k in range(max_goals + 1)], dtype=float)
    return vals


def poisson_outcome(lambda_a, lambda_b, max_goals=MAX_GOALS):
    a = poisson_pmf(lambda_a, max_goals)
    b = poisson_pmf(lambda_b, max_goals)
    grid = np.outer(a, b)
    grid = grid / grid.sum()
    rows = []
    for i in range(max_goals + 1):
        for j in range(max_goals + 1):
            rows.append({"argentina_goals": i, "england_goals": j, "probability": float(grid[i, j])})
    scorelines = pd.DataFrame(rows).sort_values("probability", ascending=False)
    return {
        "argentina_win_90": float(np.tril(grid, -1).sum()),
        "draw_90": float(np.trace(grid)),
        "england_win_90": float(np.triu(grid, 1).sum()),
    }, scorelines


## Load local shots

The shot dataset excludes period 5 shootout shots. StatsBomb xG is stored for benchmarking only and is not used as a custom model feature.

In [ ]:

competitions = json.loads((DATA_DIR / "competitions.json").read_text(encoding="utf-8"))
selected_competitions = []
for comp in competitions:
    key = (str(comp.get("competition_name")), str(comp.get("season_name")))
    if key in SELECTED_COMPETITIONS and str(comp.get("competition_gender", "")).lower() == "male":
        selected_competitions.append(comp)

matches = []
for comp in selected_competitions:
    path = DATA_DIR / "matches" / str(comp["competition_id"]) / f"{comp['season_id']}.json"
    if not path.exists():
        continue
    for match in json.loads(path.read_text(encoding="utf-8")):
        matches.append({
            "match_id": match["match_id"],
            "competition_name": comp["competition_name"],
            "season_name": comp["season_name"],
            "match_date": match.get("match_date"),
            "home_team": get_nested(match, ["home_team", "home_team_name"], default=""),
            "away_team": get_nested(match, ["away_team", "away_team_name"], default=""),
        })
match_meta = pd.DataFrame(matches)
match_id_to_meta = match_meta.set_index("match_id").to_dict(orient="index")

shot_rows = []
for match_id in match_meta["match_id"].tolist():
    event_path = DATA_DIR / "events" / f"{match_id}.json"
    if not event_path.exists():
        continue
    events = json.loads(event_path.read_text(encoding="utf-8"))
    meta = match_id_to_meta[match_id]
    for event in events:
        if get_nested(event, ["type", "name"], default="") != "Shot":
            continue
        period = int(event.get("period", 0) or 0)
        if period == 5:
            continue
        loc = event.get("location") or [np.nan, np.nan]
        if len(loc) < 2:
            continue
        x, y = float(loc[0]), float(loc[1])
        teammates, opponents, opponents_goal_side, min_opp_dist, gk_dist = freeze_features(event)
        outcome = get_nested(event, ["shot", "outcome", "name"], default="")
        row = {
            "match_id": match_id,
            "team": get_nested(event, ["team", "name"], default=""),
            "player": get_nested(event, ["player", "name"], default=""),
            "competition_name": meta["competition_name"],
            "season_name": meta["season_name"],
            "match_date": meta["match_date"],
            "home_team": meta["home_team"],
            "away_team": meta["away_team"],
            "period": period,
            "minute": int(event.get("minute", 0) or 0),
            "x": x,
            "y": y,
            "distance_to_goal": float(math.hypot(120.0 - x, 40.0 - y)),
            "angle_to_goal": shot_angle(x, y),
            "shot_type": get_nested(event, ["shot", "type", "name"], default="Unknown"),
            "body_part": get_nested(event, ["shot", "body_part", "name"], default="Unknown"),
            "technique": get_nested(event, ["shot", "technique", "name"], default="Unknown"),
            "play_pattern": get_nested(event, ["play_pattern", "name"], default="Unknown"),
            "first_time": bool(get_nested(event, ["shot", "first_time"], default=False)),
            "aerial_won": bool(get_nested(event, ["shot", "aerial_won"], default=False)),
            "one_on_one": bool(get_nested(event, ["shot", "one_on_one"], default=False)),
            "open_goal": bool(get_nested(event, ["shot", "open_goal"], default=False)),
            "follows_dribble": bool(get_nested(event, ["shot", "follows_dribble"], default=False)),
            "freeze_teammates": teammates,
            "freeze_opponents": opponents,
            "freeze_opponents_goal_side": opponents_goal_side,
            "freeze_min_opponent_distance": min_opp_dist,
            "freeze_goalkeeper_distance": gk_dist,
            "is_goal": 1 if outcome == "Goal" else 0,
            "statsbomb_xg": float(get_nested(event, ["shot", "statsbomb_xg"], default=np.nan)),
        }
        shot_rows.append(row)

shots = pd.DataFrame(shot_rows)
if shots.empty:
    raise ValueError("No shots found in selected local StatsBomb events")
shots = shots[~shots["team"].astype(str).str.contains("Women|Women's", case=False, na=False)].copy()
shots.to_csv(OUTPUT_DIR / "custom_xg_shot_dataset.csv", index=False)

print("Selected competitions:")
print(pd.DataFrame(selected_competitions)[["competition_name", "season_name", "competition_id", "season_id"]].to_string(index=False))
print("Shot dataset:", shots.shape)
shots.head()


## Train custom xG

The custom model is a leakage-safe supervised xG model using shot location, angle, body part, play pattern, timing, and freeze-frame pressure features. The evaluation split is grouped by match ID.

In [ ]:

numeric_features = [
    "x", "y", "distance_to_goal", "angle_to_goal", "period", "minute",
    "freeze_teammates", "freeze_opponents", "freeze_opponents_goal_side",
    "freeze_min_opponent_distance", "freeze_goalkeeper_distance",
]
categorical_features = [
    "shot_type", "body_part", "technique", "play_pattern",
    "first_time", "aerial_won", "one_on_one", "open_goal", "follows_dribble",
]
leakage_fields = ["statsbomb_xg", "is_goal"]
assert "statsbomb_xg" not in numeric_features + categorical_features
assert "is_goal" not in numeric_features + categorical_features

model_df = shots.dropna(subset=["is_goal"]).copy()
groups = model_df["match_id"]
y = model_df["is_goal"].astype(int)
X = model_df[numeric_features + categorical_features]

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))
assert set(model_df.iloc[train_idx]["match_id"]).isdisjoint(set(model_df.iloc[test_idx]["match_id"]))

preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numeric_features),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", make_ohe())]), categorical_features),
    ],
    remainder="drop",
)
clf = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=220,
    max_leaf_nodes=18,
    l2_regularization=0.02,
    random_state=42,
)
model = Pipeline([("preprocess", preprocess), ("classifier", clf)])
model.fit(X.iloc[train_idx], y.iloc[train_idx])

test_pred = model.predict_proba(X.iloc[test_idx])[:, 1]
test_actual = y.iloc[test_idx].to_numpy()
test_sb = model_df.iloc[test_idx]["statsbomb_xg"].astype(float).clip(1e-6, 1 - 1e-6).fillna(model_df["is_goal"].mean()).to_numpy()

metrics = pd.DataFrame([
    {
        "model": "custom_hist_gradient_boosting",
        "log_loss": log_loss(test_actual, np.clip(test_pred, 1e-6, 1 - 1e-6)),
        "brier_score": brier_score_loss(test_actual, test_pred),
        "roc_auc": roc_auc_score(test_actual, test_pred) if len(np.unique(test_actual)) > 1 else np.nan,
        "test_shots": len(test_idx),
        "train_matches": model_df.iloc[train_idx]["match_id"].nunique(),
        "test_matches": model_df.iloc[test_idx]["match_id"].nunique(),
    },
    {
        "model": "statsbomb_xg_benchmark",
        "log_loss": log_loss(test_actual, test_sb),
        "brier_score": brier_score_loss(test_actual, test_sb),
        "roc_auc": roc_auc_score(test_actual, test_sb) if len(np.unique(test_actual)) > 1 else np.nan,
        "test_shots": len(test_idx),
        "train_matches": model_df.iloc[train_idx]["match_id"].nunique(),
        "test_matches": model_df.iloc[test_idx]["match_id"].nunique(),
    },
])
metrics.to_csv(OUTPUT_DIR / "custom_xg_model_metrics.csv", index=False)

calib = pd.DataFrame({"actual": test_actual, "custom_xg": test_pred, "statsbomb_xg": test_sb})
calib["custom_bin"] = pd.qcut(calib["custom_xg"].rank(method="first"), q=min(10, len(calib)), labels=False)
calibration = calib.groupby("custom_bin").agg(
    shots=("actual", "count"),
    mean_custom_xg=("custom_xg", "mean"),
    mean_statsbomb_xg=("statsbomb_xg", "mean"),
    actual_goal_rate=("actual", "mean"),
).reset_index()
calibration.to_csv(OUTPUT_DIR / "custom_xg_calibration_table.csv", index=False)

plt.figure(figsize=(6, 5))
plt.plot(calibration["mean_custom_xg"], calibration["actual_goal_rate"], marker="o", label="Custom xG")
plt.plot(calibration["mean_statsbomb_xg"], calibration["actual_goal_rate"], marker="s", label="StatsBomb xG")
mx = max(calibration[["mean_custom_xg", "mean_statsbomb_xg", "actual_goal_rate"]].max().max(), 0.05)
plt.plot([0, mx], [0, mx], linestyle="--", color="black", linewidth=1, label="Perfect calibration")
plt.xlabel("Mean predicted xG")
plt.ylabel("Actual goal rate")
plt.title("Custom xG calibration")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "custom_xg_calibration_plot.png", dpi=180)
plt.close()

sample_n = min(1200, len(test_idx))
sample_X = X.iloc[test_idx].sample(sample_n, random_state=42)
sample_y = y.loc[sample_X.index]
perm = permutation_importance(model, sample_X, sample_y, n_repeats=3, random_state=42, scoring="neg_log_loss")
importance = pd.DataFrame({
    "feature": numeric_features + categorical_features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
})
importance["reinforced_from_spain_france_custom_xg"] = importance["feature"].isin(REINFORCED_FEATURE_HINTS)
importance = importance.sort_values("importance_mean", ascending=False)
importance.to_csv(OUTPUT_DIR / "custom_xg_feature_importance.csv", index=False)

plot_imp = importance.head(12).sort_values("importance_mean")
plt.figure(figsize=(8, 5))
colors = ["#4c78a8" if not x else "#f58518" for x in plot_imp["reinforced_from_spain_france_custom_xg"]]
plt.barh(plot_imp["feature"], plot_imp["importance_mean"], color=colors)
plt.xlabel("Permutation importance, neg log loss")
plt.title("Custom xG feature importance")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "custom_xg_feature_importance.png", dpi=180)
plt.close()

shots["custom_xg"] = model.predict_proba(shots[numeric_features + categorical_features])[:, 1]
shots["custom_xg"] = shots["custom_xg"].clip(0, 1)
shots.to_csv(OUTPUT_DIR / "custom_xg_shot_dataset_with_predictions.csv", index=False)
metrics


## Project pre-match xG

The projection uses recent weighted xG for and against, shrinkage toward the global tournament average, a neutral-site assumption, and an independent Poisson score table.

In [ ]:

def make_team_match_dataset(xg_column, periods=(1, 2)):
    use = shots[shots["period"].isin(periods)].copy()
    rows = []
    for _, match in match_meta.iterrows():
        match_id = match["match_id"]
        mshots = use[use["match_id"].eq(match_id)]
        teams = [match["home_team"], match["away_team"]]
        if not teams[0] or not teams[1]:
            teams = sorted(mshots["team"].dropna().unique().tolist())
        if len(teams) < 2:
            continue
        for team in teams[:2]:
            opponent = teams[1] if team == teams[0] else teams[0]
            for_shots = mshots[mshots["team"].eq(team)]
            against_shots = mshots[mshots["team"].eq(opponent)]
            rows.append({
                "match_id": match_id,
                "team": team,
                "opponent": opponent,
                "competition_name": match["competition_name"],
                "season_name": match["season_name"],
                "match_date": match["match_date"],
                "xg_for": float(for_shots[xg_column].sum()),
                "xg_against": float(against_shots[xg_column].sum()),
                "shots_for": int(len(for_shots)),
                "shots_against": int(len(against_shots)),
                "goals_for": int(for_shots["is_goal"].sum()),
                "goals_against": int(against_shots["is_goal"].sum()),
                "xg_source": xg_column,
            })
    out = pd.DataFrame(rows)
    out["match_date"] = pd.to_datetime(out["match_date"], errors="coerce")
    return out.sort_values(["match_date", "match_id", "team"]).reset_index(drop=True)


def weighted_recent_average(values):
    values = np.asarray(values, dtype=float)
    if len(values) == 0:
        return np.nan
    weights = np.arange(1, len(values) + 1, dtype=float)
    return float(np.average(values, weights=weights))


def project_match(team_dataset, xg_source_name):
    global_avg = float(team_dataset["xg_for"].mean())
    inputs = []
    ratings = {}
    for team in [TEAM_A, TEAM_B]:
        history = team_dataset[team_dataset["team"].eq(team)].sort_values("match_date").tail(RECENT_MATCH_COUNT)
        if history.empty:
            raise ValueError(f"No xG history found for {team} using {xg_source_name}")
        recent_xgf = weighted_recent_average(history["xg_for"])
        recent_xga = weighted_recent_average(history["xg_against"])
        matches = len(history)
        attack_raw = recent_xgf / global_avg if global_avg else 1.0
        defense_raw = recent_xga / global_avg if global_avg else 1.0
        attack_rating = (matches * attack_raw + SHRINKAGE_MATCHES * 1.0) / (matches + SHRINKAGE_MATCHES)
        defense_rating = (matches * defense_raw + SHRINKAGE_MATCHES * 1.0) / (matches + SHRINKAGE_MATCHES)
        ratings[team] = {"attack": attack_rating, "defense": defense_rating}
        inputs.append({
            "xg_source": xg_source_name,
            "team": team,
            "matches_used": matches,
            "recent_weighted_xg_for": recent_xgf,
            "recent_weighted_xg_against": recent_xga,
            "global_average_team_xg": global_avg,
            "attack_rating_shrunk": attack_rating,
            "defense_rating_shrunk": defense_rating,
            "latest_match_date_in_history": history["match_date"].max(),
        })
    lambda_a = float(global_avg * ratings[TEAM_A]["attack"] * ratings[TEAM_B]["defense"])
    lambda_b = float(global_avg * ratings[TEAM_B]["attack"] * ratings[TEAM_A]["defense"])
    probs, scorelines = poisson_outcome(lambda_a, lambda_b)
    prediction = {
        "xg_source": xg_source_name,
        "match": MATCH_LABEL,
        "match_date": MATCH_DATE,
        "argentina_projected_xg": lambda_a,
        "england_projected_xg": lambda_b,
        **probs,
    }
    prediction["most_likely_scoreline"] = f"{int(scorelines.iloc[0]['argentina_goals'])}-{int(scorelines.iloc[0]['england_goals'])}"
    prediction["predicted_90_minute_result"] = max(
        [("Argentina win", prediction["argentina_win_90"]), ("Draw", prediction["draw_90"]), ("England win", prediction["england_win_90"])],
        key=lambda x: x[1],
    )[0]
    scorelines["xg_source"] = xg_source_name
    return pd.DataFrame(inputs), pd.DataFrame([prediction]), scorelines


statsbomb_team = make_team_match_dataset("statsbomb_xg")
custom_team = make_team_match_dataset("custom_xg")
custom_team.to_csv(OUTPUT_DIR / "custom_xg_team_match_dataset.csv", index=False)

sb_inputs, sb_pred, sb_scores = project_match(statsbomb_team, "statsbomb_xg")
custom_inputs, custom_pred, custom_scores = project_match(custom_team, "custom_xg")

inputs = pd.concat([sb_inputs, custom_inputs], ignore_index=True)
inputs.to_csv(OUTPUT_DIR / "argentina_england_custom_vs_statsbomb_xg_inputs.csv", index=False)

comparison = pd.concat([sb_pred, custom_pred], ignore_index=True)
comparison.to_csv(OUTPUT_DIR / "argentina_england_statsbomb_vs_custom_xg_comparison.csv", index=False)
custom_pred.to_csv(OUTPUT_DIR / "argentina_england_custom_xg_prediction.csv", index=False)
custom_scores.to_csv(OUTPUT_DIR / "argentina_england_custom_xg_scoreline_probabilities.csv", index=False)

diff = {
    "comparison": "custom_minus_statsbomb",
    "argentina_projected_xg_diff": float(custom_pred.iloc[0]["argentina_projected_xg"] - sb_pred.iloc[0]["argentina_projected_xg"]),
    "england_projected_xg_diff": float(custom_pred.iloc[0]["england_projected_xg"] - sb_pred.iloc[0]["england_projected_xg"]),
    "argentina_win_90_diff": float(custom_pred.iloc[0]["argentina_win_90"] - sb_pred.iloc[0]["argentina_win_90"]),
    "draw_90_diff": float(custom_pred.iloc[0]["draw_90"] - sb_pred.iloc[0]["draw_90"]),
    "england_win_90_diff": float(custom_pred.iloc[0]["england_win_90"] - sb_pred.iloc[0]["england_win_90"]),
}
pd.DataFrame([diff]).to_csv(OUTPUT_DIR / "argentina_england_statsbomb_vs_custom_xg_differences.csv", index=False)

comparison


## Interpretation and saved outputs

The report compares custom xG and StatsBomb xG outputs, and carries forward the Spain-France feature lesson without overfitting to a single result.

In [ ]:

plot_comp = comparison.copy()
plt.figure(figsize=(7, 4))
x = np.arange(len(plot_comp))
width = 0.35
plt.bar(x - width / 2, plot_comp["argentina_projected_xg"], width, label="Argentina", color="#4c78a8")
plt.bar(x + width / 2, plot_comp["england_projected_xg"], width, label="England", color="#f58518")
plt.xticks(x, plot_comp["xg_source"])
plt.ylabel("Projected xG")
plt.title("Projected xG by model source")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "statsbomb_vs_custom_projected_xg.png", dpi=180)
plt.close()

plt.figure(figsize=(8, 4))
labels = ["Argentina win", "Draw", "England win"]
for _, row in plot_comp.iterrows():
    vals = [row["argentina_win_90"], row["draw_90"], row["england_win_90"]]
    plt.plot(labels, vals, marker="o", label=row["xg_source"])
plt.ylabel("90-minute probability")
plt.title("Outcome probabilities by xG source")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "statsbomb_vs_custom_outcome_probabilities.png", dpi=180)
plt.close()

heat = custom_scores.pivot(index="argentina_goals", columns="england_goals", values="probability").sort_index(ascending=True)
plt.figure(figsize=(7, 6))
plt.imshow(heat.values, cmap="magma", origin="lower")
plt.colorbar(label="Probability")
plt.xticks(range(len(heat.columns)), heat.columns)
plt.yticks(range(len(heat.index)), heat.index)
plt.xlabel("England goals")
plt.ylabel("Argentina goals")
plt.title("Custom xG Poisson scoreline heatmap")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "custom_xg_scoreline_heatmap.png", dpi=180)
plt.close()

important_reinforced = importance[importance["reinforced_from_spain_france_custom_xg"]].head(8)
custom_row = custom_pred.iloc[0]
sb_row = sb_pred.iloc[0]

report = f"""# Argentina vs England xG Prediction

Match date: {MATCH_DATE}

This notebook trains a local custom xG model from raw StatsBomb shot events and compares it with the StatsBomb-provided `shot.statsbomb_xg` benchmark. It uses local JSON only and does not use current 2026 event data, odds, media commentary, or confirmed lineups.

## Spain-France feedback

{POST_MATCH_FEEDBACK}

I do not add that single match as a training label. Instead, the notebook reinforces the lesson from the prior custom xG experiment by explicitly tracking features that helped explain shot quality: angle to goal, defensive pressure/freezeframe spacing, goalkeeper distance, shot location, body part, open goal, and play pattern.

## Custom xG headline prediction

- Argentina projected xG: {custom_row['argentina_projected_xg']:.3f}
- England projected xG: {custom_row['england_projected_xg']:.3f}
- Argentina win in 90 minutes: {custom_row['argentina_win_90']:.1%}
- Draw after 90 minutes: {custom_row['draw_90']:.1%}
- England win in 90 minutes: {custom_row['england_win_90']:.1%}
- Most likely scoreline: {custom_row['most_likely_scoreline']}

## StatsBomb xG benchmark

- Argentina projected xG: {sb_row['argentina_projected_xg']:.3f}
- England projected xG: {sb_row['england_projected_xg']:.3f}
- Argentina win in 90 minutes: {sb_row['argentina_win_90']:.1%}
- Draw after 90 minutes: {sb_row['draw_90']:.1%}
- England win in 90 minutes: {sb_row['england_win_90']:.1%}
- Most likely scoreline: {sb_row['most_likely_scoreline']}

## Model caveat

StatsBomb xG is the vendor-provided supervised xG benchmark. The custom model is educational: it is trained on available open-data shots and evaluated with a grouped match split so shots from the same match do not appear in both train and test.
"""
(OUTPUT_DIR / "argentina_england_xg_report.md").write_text(report, encoding="utf-8")

checklist = pd.DataFrame([
    {"check": "output_folder_created", "status": OUTPUT_DIR.exists()},
    {"check": "period_5_excluded", "status": bool((shots["period"] == 5).sum() == 0)},
    {"check": "statsbomb_xg_not_a_feature", "status": "statsbomb_xg" not in numeric_features + categorical_features},
    {"check": "group_split_by_match", "status": bool(set(model_df.iloc[train_idx]["match_id"]).isdisjoint(set(model_df.iloc[test_idx]["match_id"])))},
    {"check": "custom_xg_between_0_and_1", "status": bool(shots["custom_xg"].between(0, 1).all())},
    {"check": "poisson_probs_sum_to_one", "status": bool(np.allclose(comparison[["argentina_win_90", "draw_90", "england_win_90"]].sum(axis=1), 1.0))},
])
checklist.to_csv(OUTPUT_DIR / "final_checklist.csv", index=False)

print("Completed xG Argentina vs England notebook.")
print("Output folder:", OUTPUT_DIR)
print(comparison.to_string(index=False))
